# Lipi: Remote Experiment Runner (Kaggle & Cloud GPUs)

Optimized for **Kaggle GPU** (T4 x2 or P100), **Google Colab**, and remote Jupyter environments.

### Quick Setup on Kaggle:
1. **Accelerator**: Select **GPU T4 x2** or **P100** in the right-hand **Notebook options** sidebar.
2. **Internet**: Toggle **Internet ON** in the right-hand sidebar (required to download code & Zenodo data).
3. **Shard Selection**: Set `SHARD_INDEX` below (1 to 12) for this Kaggle account / session.
4. **Run All Cells**: Press `Shift+Enter` or click **Run All**.
5. **Download**: When training completes, your result zip will appear in the right-hand **Output** pane for 1-click download.

In [ ]:
# ==========================================================================
# 1. EXPERIMENT SHARD CONFIGURATION
# ==========================================================================
REPO_URL     = "https://github.com/ashimst/lipi.git"
DATA_URL     = "https://zenodo.org/records/24240673/files/lipi_dataset_v1.0.zip?download=1"

# Choose your shard index (1 through 12). Each shard has exactly 16 jobs (~1.5 to 2 hrs on GPU T4).
# Run different shards on different Kaggle accounts or sessions to complete all 192 jobs quickly.
SHARD_INDEX  = 1          # <-- CHANGE THIS (1 to 12) FOR EACH SESSION
TOTAL_SHARDS = 12

WHO  = f"shard_{SHARD_INDEX:02d}"
WHAT = f"--shard {SHARD_INDEX}/{TOTAL_SHARDS}"

# Alternative: Target specific experiment groups directly (uncomment if preferred):
# WHO, WHAT = "loso_p1",       "--group loso --shard 1/2"
# WHO, WHAT = "loso_p2",       "--group loso --shard 2/2"
# WHO, WHAT = "encoders_p1",   "--group encoders --shard 1/2"
# WHO, WHAT = "encoders_p2",   "--group encoders --shard 2/2"
# WHO, WHAT = "baselines",     "--group baselines"
# WHO, WHAT = "resolution_p1", "--group resolution --shard 1/3"
# WHO, WHAT = "resolution_p2", "--group resolution --shard 2/3"
# WHO, WHAT = "resolution_p3", "--group resolution --shard 3/3"
# WHO, WHAT = "single_p1",     "--group single --shard 1/2"
# WHO, WHAT = "single_p2",     "--group single --shard 2/2"
# WHO, WHAT = "within",        "--group within"

KEEP_RUNS_IN = ""  # Optional: persistent Google Drive path if on Colab (e.g. "/content/drive/MyDrive/lipi_runs")

In [ ]:
# ==========================================================================
# 2. ENVIRONMENT SETUP & REPOSITORY INITIALIZATION
# ==========================================================================
import os, subprocess, sys
from pathlib import Path

# On Kaggle, ensure workspace begins at /kaggle/working
if os.path.exists("/kaggle/working"):
    os.chdir("/kaggle/working")

# Clone repo if not already cloned; avoid nested directories on re-execution
if Path.cwd().name != "lipi":
    if not (Path.cwd() / "lipi").exists():
        print("Cloning repository...")
        subprocess.run(["git", "clone", REPO_URL, "lipi"], check=True)
    os.chdir("lipi")

# Pull latest changes and install dependencies
subprocess.run(["git", "pull"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import torch
if torch.cuda.is_available():
    print(f"✓ GPU Ready: {torch.cuda.get_device_name(0)}")
else:
    print("⚠ WARNING: No GPU detected! Go to Kaggle sidebar -> Accelerator -> GPU T4 x2 or P100")

In [ ]:
# ==========================================================================
# 3. DATASET DOWNLOAD FROM ZENODO (AUTOMATIC CACHING)
# ==========================================================================
if not os.path.exists("data/canonical"):
    kaggle_input = Path("/kaggle/input")
    found = list(kaggle_input.glob("**/canonical")) if kaggle_input.exists() else []
    
    if found:
        print(f"Found attached Kaggle dataset at: {found[0]}")
        os.makedirs("data", exist_ok=True)
        try:
            os.symlink(str(found[0]), "data/canonical")
        except OSError:
            import shutil
            shutil.copytree(str(found[0]), "data/canonical")
    elif DATA_URL:
        print(f"Downloading dataset from Zenodo ({DATA_URL})...")
        subprocess.run(["curl", "-L", "-o", "lipi_data.zip", DATA_URL], check=True)
        print("Extracting dataset into data/...")
        subprocess.run(["unzip", "-q", "lipi_data.zip", "-d", "data/"], check=True)
        if os.path.exists("lipi_data.zip"):
            os.remove("lipi_data.zip")

assert os.path.exists("data/canonical/brahmi"), (
    "ERROR: data/canonical not populated! Check that Internet access is toggled ON in the Kaggle sidebar."
)
print("✓ Dataset verified: 16,350 canonical glyph images available across 7 historical scripts.")

In [ ]:
# ==========================================================================
# 4. SMOKE TEST (END-TO-END SANITY CHECK)
# ==========================================================================
if KEEP_RUNS_IN:
    if KEEP_RUNS_IN.startswith("/content/drive"):
        from google.colab import drive; drive.mount("/content/drive")
    os.makedirs(KEEP_RUNS_IN, exist_ok=True)
    os.environ["LIPI_RUNS_DIR"] = KEEP_RUNS_IN

!python run.py smoke

In [ ]:
# ==========================================================================
# 5. INSPECT SHARD JOBS & EXECUTE TRAINING
# ==========================================================================
print(f"=== ASSIGNMENT FOR {WHO} ({WHAT}) ===\n")
!python run.py list {WHAT}
print(f"\nStarting training runs for {WHO}...")
!python run.py train {WHAT} --who {WHO}

In [ ]:
# ==========================================================================
# 6. VERIFY & PACK OUTPUT ZIP FOR DOWNLOAD
# ==========================================================================
from pathlib import Path

!python run.py status

# Write output archive to /kaggle/working so it is immediately visible in Kaggle UI
out_dir = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
!python run.py pack --who {WHO} --out {out_dir}

# Find output archive
archives = sorted(Path(out_dir).glob(f"lipi_runs_{WHO}_*.zip"))
if archives:
    latest = archives[-1]
    size_mb = latest.stat().st_size / (1024 * 1024)
    print(f"\n" + "="*60)
    print(f"✓ SUCCESS: Bundle created: {latest.name} ({size_mb:.1f} MB)")
    print(f"  Location: {latest.resolve()}")
    print("  -> Download this file from the Kaggle Output panel on the right.")
    try:
        from IPython.display import FileLink
        display(FileLink(str(latest)))
    except Exception:
        pass
    print("="*60)
else:
    print("Notice: No completed jobs found to pack.")